### Load the Anthropic API key

In [29]:
from dotenv import load_dotenv

load_dotenv()

True

### Define the anthropic client and model

In [30]:
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-5"

### First API call

In [31]:
message = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "hello, my name is Juan David"
        }
    ]
)


In [32]:
message.content[0].text

'Hello Juan David! Nice to meet you. How can I help you today?'

### Let's try to have a conversation

In [33]:
message = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "What is my name?"
        }
    ]
)

In [34]:
message.content[0].text

"I don't know your name - you haven't told me yet! Would you like to share it with me?"

### Let's actually build the conversation by storing the whole list of messages

#### Create helper functions

In [35]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)
    return messages

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)
    return messages

def chat(messages, system=None, temperature=1.0, stop_sequences=None):

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(
        **params
    )
    return message.content[0].text

#### Set the system for having a conversation

In [36]:
# Initialize messages list

messages = []

messages = add_user_message(messages, "Hi, my name is Juan David")

response_assistant = chat(messages)

messages = add_assistant_message(messages, response_assistant)

messages = add_user_message(messages, "What is my name?")

response_assistant = chat(messages)

messages = add_assistant_message(messages, response_assistant)

In [37]:
messages

[{'role': 'user', 'content': 'Hi, my name is Juan David'},
 {'role': 'assistant',
  'content': 'Hello Juan David! Nice to meet you. How can I help you today?'},
 {'role': 'user', 'content': 'What is my name?'},
 {'role': 'assistant',
  'content': 'Your name is Juan David. Is there anything I can help you with today?'}]

## Create a chatbot (Uncomment if you want to test it)

In [38]:
# # Get input from the user:

# messages = []

# while True:

#     user_message = input("type something")
#     print(f"user: {user_message}")
#     messages = add_user_message(messages, user_message)
#     assistant_message = chat(messages)
#     print(f"assistant: {assistant_message}")
#     messages = add_assistant_message(messages, assistant_message)

## Now test a system prompt

#### Raw claude call

In [39]:
messages = []

messages = add_user_message(
    messages, "how to solve 3x+2=5"
)

assistant_response = chat(messages)

print(assistant_response)

# How to solve 3x + 2 = 5

**Goal:** Isolate x on one side of the equation

## Step-by-step solution:

**Step 1:** Subtract 2 from both sides
```
3x + 2 = 5
3x + 2 - 2 = 5 - 2
3x = 3
```

**Step 2:** Divide both sides by 3
```
3x = 3
3x/3 = 3/3
x = 1
```

## Answer: **x = 1**

## Check your work:
Substitute x = 1 back into the original equation:
```
3(1) + 2 = 5
3 + 2 = 5
5 = 5 ✓
```

The solution is correct!


#### Adding a system prompt acting as a tutor

In [40]:
system_prompt = """
You're math tutor, only give hints to the student. Do not
give the answer to the student, guide him.
"""

messages = []

messages = add_user_message(messages, "how to solve 3x+2=5")

assistant_response = chat(messages, system=system_prompt)

print(assistant_response)

Great! Let me help you solve this equation step by step.

The goal is to get **x by itself** on one side of the equation.

Right now you have: 3x + 2 = 5

**First hint:** What do you notice is "attached" to the x term? There's a 3 being multiplied and a 2 being added.

Which operation should you undo first - the adding 2, or the multiplying by 3?

Think about the order: what's happening closest to x, and what's happening further away?


## Exercise

#### Raw claude call

In [41]:
messages = []

messages = add_user_message(
    messages,
    "Write a Python function that checks a string for duplicate characters"
)

assistant_response = chat(messages)

print(assistant_response)

# Check String for Duplicate Characters

Here are several approaches to check for duplicate characters in a string:

## 1. Simple Boolean Check

```python
def has_duplicates(s):
    """
    Check if a string contains duplicate characters.
    
    Args:
        s: String to check
        
    Returns:
        bool: True if duplicates exist, False otherwise
    """
    return len(s) != len(set(s))

# Example usage
print(has_duplicates("hello"))      # True (l appears twice)
print(has_duplicates("world"))      # True (o appears twice)
print(has_duplicates("python"))     # False
```

## 2. Find All Duplicate Characters

```python
def find_duplicates(s):
    """
    Find all duplicate characters in a string.
    
    Args:
        s: String to check
        
    Returns:
        set: Set of duplicate characters
    """
    seen = set()
    duplicates = set()
    
    for char in s:
        if char in seen:
            duplicates.add(char)
        else:
            seen.add(char)
    
    r

#### Now with system prompt

In [42]:
system_prompt = """
You're a staff software engineer that answers as concisely as possible, only give the code to the user, do not explain anything.
"""

messages = []

messages = add_user_message(
    messages,
    "Write a Python function that checks a string for duplicate characters"
)

assistant_response = chat(messages, system=system_prompt)

print(assistant_response)


```python
def has_duplicates(s: str) -> bool:
    return len(s) != len(set(s))
```


## Let's experiment with temperature

In [43]:
messages = []

messages = add_user_message(messages, "Generate a one sentence movie idea")

answer = chat(messages, temperature=1)

print(answer)

A grieving astronaut discovers that the mysterious signal she's been tracking from deep space is actually a message from her own future self, warning her not to return to Earth.


In [44]:
messages = []

messages = add_user_message(messages, "Generate a one sentence movie idea")

answer = chat(messages, temperature=0)

print(answer)

A retired astronaut discovers her childhood imaginary friend was actually an alien who has returned to warn her that Earth is about to pass through a cosmic phenomenon that will make all imaginary friends become real.


In [45]:
messages = []

messages = add_user_message(messages, "Generate a one sentence movie idea")

answer = chat(messages, temperature=0)

print(answer)

A retired astronaut discovers her childhood imaginary friend was actually an alien who has returned to warn her that Earth is about to pass through a cosmic phenomenon that will make all imaginary friends become real.


## Streaming

In [46]:
messages = []

messages = add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

In [47]:
for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_016WreD3Z9oHGKqcxBmm8wiu', container=None, content=[], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=6, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A fake database is a sim', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='ulated or mock data storage system that mimics the structure and behavior of a real database but contains', type='text_delta'), index=0, type='co

## Simpler streaming with claude python SDK

In [48]:
messages = []

messages = add_user_message(messages, "Write a one sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
    temperature=0
) as stream:
    for text in stream.text_stream:
        pass

    final_message = stream.get_final_message()

## Structured data

In [49]:
messages = []

messages = add_user_message(messages, "Generate a very short event bridge rule as json")
mesages = add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

In [50]:
text

'\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n'

In [51]:
text.strip()

'{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}'

In [52]:
import json 
json.loads(text.strip())

{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}

## Structured data exercise

In [53]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short
"""

messages = add_user_message(messages, prompt)
messages = add_assistant_message(messages, "Here are all three commands in a single block without any comments: \n```bash")

text = chat(messages, stop_sequences=["```"])

text.strip()

'aws s3 ls\n\naws ec2 describe-instances\n\naws iam list-users'

## Prompt evaluation

In [54]:
def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
                "format": "json" or "python" or "regex",
                "solution_criteria": "Key criteria for evaluating the solution"
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """

    messages = []

    messages = add_user_message(messages, prompt)
    messages = add_assistant_message(messages, "```json")

    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)

In [55]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [56]:
dataset

[{'task': 'Write a regular expression that validates AWS S3 bucket names according to AWS naming rules: 3-63 characters long, only lowercase letters, numbers, hyphens, and periods, must start and end with a letter or number, and cannot contain consecutive periods or period-hyphen combinations.',
  'format': 'regex',
  'solution_criteria': "Regex must correctly validate valid bucket names (e.g., 'my-bucket-123', 'example.bucket') and reject invalid ones (e.g., 'My-Bucket', 'a', 'bucket..name', 'bucket.-name', '-bucket')"},
 {'task': "Create a JSON policy document that grants read-only access to a specific S3 bucket named 'company-logs' for an IAM user, allowing them to list objects and get objects but not delete or modify them.",
  'format': 'json',
  'solution_criteria': "JSON must be valid IAM policy syntax with Version, Statement array containing Effect 'Allow', appropriate Actions (s3:GetObject, s3:ListBucket), and Resource ARN pointing to 'company-logs' bucket"},
 {'task': "Write a

## Implement a grading system

In [57]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
        You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

        Original Task:
        <task>
        {test_case["task"]}
        </task>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Criteria you should use to evaluate the solution:
        <solution criteria>
        {test_case["solution_criteria"]}
        </solution criteria>

        Output Format
        Provide your evaluation as a structured JSON object with the following fields, in this specific order:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "score": A number between 1-10

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
    """

    messages = []
    messages = add_user_message(messages, eval_prompt)
    messages = add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [58]:
def run_prompt(test_case):
    messages = []
    prompt = f"""
    You're a Staff Software engineer at Anthropic, ex OpenAI, ex Google, ex Meta with 25 years of experience.
    You're the best software engineer in the world. NO MISTAKES IN ANY SYNTAX ALLOWED.
    
    Please solve the following task:
        
        {test_case["task"]}

    * Respond only with Python, JSON, or a plain regex
    * Do not add any comments or comentary or explanation
    """
    messages = add_user_message(messages, prompt)
    messages = add_assistant_message(messages, "```code")
    text = chat(messages, stop_sequences=["```"])
    return text

In [59]:
def run_test_case(test_case):

    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [60]:
from statistics import mean

def run_eval(dataset):

    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        print(f"Score: {result["score"]}")
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    return results

In [61]:
results = run_eval(dataset)

Score: 8
Score: 9
Score: 7
Average score: 8


In [62]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n^(?!.*\\.\\.)(?!.*\\.\\-)(?!.*\\-\\.)[a-z0-9][a-z0-9.\\-]{1,61}[a-z0-9]$\n",
    "test_case": {
      "task": "Write a regular expression that validates AWS S3 bucket names according to AWS naming rules: 3-63 characters long, only lowercase letters, numbers, hyphens, and periods, must start and end with a letter or number, and cannot contain consecutive periods or period-hyphen combinations.",
      "format": "regex",
      "solution_criteria": "Regex must correctly validate valid bucket names (e.g., 'my-bucket-123', 'example.bucket') and reject invalid ones (e.g., 'My-Bucket', 'a', 'bucket..name', 'bucket.-name', '-bucket')"
    },
    "score": 8,
    "reasoning": "The regex correctly implements most core AWS S3 bucket naming rules including length (3-63 characters), character restrictions, start/end requirements, and prevention of consecutive periods and period-hyphen combinations. It successfully validates valid bucket names and rejects most invalid ones. Howe

## Now code grader

In [63]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format_test = test_case["format"]
    if format_test == "json":
        return validate_json(response)
    elif format_test == "python":
        return validate_python(response)
    elif format_test == "regex":
        return validate_regex(response)
    else:
        raise ValueError(f"{format_test} is not allowed")

In [64]:
def run_test_case(test_case):

    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    syntax_score = grade_syntax(output, test_case)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [65]:
results = run_eval(dataset)

Score: 7.0
Score: 9.0
Score: 8.5
Average score: 8.166666666666666


In [66]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n^(?!.*\\.\\.)[a-z0-9]([a-z0-9]|(?<!\\.)-|(?<!-)\\.(?!-))*[a-z0-9]$|^[a-z0-9]$\n",
    "test_case": {
      "task": "Write a regular expression that validates AWS S3 bucket names according to AWS naming rules: 3-63 characters long, only lowercase letters, numbers, hyphens, and periods, must start and end with a letter or number, and cannot contain consecutive periods or period-hyphen combinations.",
      "format": "regex",
      "solution_criteria": "Regex must correctly validate valid bucket names (e.g., 'my-bucket-123', 'example.bucket') and reject invalid ones (e.g., 'My-Bucket', 'a', 'bucket..name', 'bucket.-name', '-bucket')"
    },
    "score": 7.0,
    "reasoning": "The regex demonstrates understanding of several AWS S3 naming rules (lowercase only, start/end with alphanumeric, no consecutive periods) and uses advanced regex features appropriately. However, it critically fails to enforce the 3-63 character length requirement, which is a fundamental AWS S3 

## Final version of the model evaluation pipeline

In [67]:
# Imports
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from anthropic import Anthropic
# Report Builder
def generate_prompt_evaluation_report(evaluation_results):
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 10
    pass_rate = (
        100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0
    )

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                color: #333;
            }}
            .header {{
                background-color: #f0f0f0;
                padding: 20px;
                border-radius: 5px;
                margin-bottom: 20px;
            }}
            .summary-stats {{
                display: flex;
                justify-content: space-between;
                flex-wrap: wrap;
                gap: 10px;
            }}
            .stat-box {{
                background-color: #fff;
                border-radius: 5px;
                padding: 15px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                flex-basis: 30%;
                min-width: 200px;
            }}
            .stat-value {{
                font-size: 24px;
                font-weight: bold;
                margin-top: 5px;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }}
            th {{
                background-color: #4a4a4a;
                color: white;
                text-align: left;
                padding: 12px;
            }}
            td {{
                padding: 10px;
                border-bottom: 1px solid #ddd;
                vertical-align: top;
            }}
            tr:nth-child(even) {{
                background-color: #f9f9f9;
            }}
            .output-cell {{
                white-space: pre-wrap;
            }}
            .score {{
                font-weight: bold;
                padding: 5px 10px;
                border-radius: 3px;
                display: inline-block;
            }}
            .score-high {{
                background-color: #c8e6c9;
                color: #2e7d32;
            }}
            .score-medium {{
                background-color: #fff9c4;
                color: #f57f17;
            }}
            .score-low {{
                background-color: #ffcdd2;
                color: #c62828;
            }}
            .output {{
                overflow: auto;
                white-space: pre-wrap;
            }}

            .output pre {{
                background-color: #f5f5f5;
                border: 1px solid #ddd;
                border-radius: 4px;
                padding: 10px;
                margin: 0;
                font-family: 'Consolas', 'Monaco', 'Courier New', monospace;
                font-size: 14px;
                line-height: 1.4;
                color: #333;
                box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
                overflow-x: auto;
                white-space: pre-wrap; 
                word-wrap: break-word; 
            }}

            td {{
                width: 20%;
            }}
            .score-col {{
                width: 80px;
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box">
                    <div>Total Test Cases</div>
                    <div class="stat-value">{total_tests}</div>
                </div>
                <div class="stat-box">
                    <div>Average Score</div>
                    <div class="stat-value">{avg_score:.1f} / {max_possible_score}</div>
                </div>
                <div class="stat-box">
                    <div>Pass Rate (≥7)</div>
                    <div class="stat-value">{pass_rate:.1f}%</div>
                </div>
            </div>
        </div>

        <table>
            <thead>
                <tr>
                    <th>Scenario</th>
                    <th>Prompt Inputs</th>
                    <th>Solution Criteria</th>
                    <th>Output</th>
                    <th>Score</th>
                    <th>Reasoning</th>
                </tr>
            </thead>
            <tbody>
    """

    for result in evaluation_results:
        prompt_inputs_html = "<br>".join(
            [
                f"<strong>{key}:</strong> {value}"
                for key, value in result["test_case"]["prompt_inputs"].items()
            ]
        )

        criteria_string = "<br>• ".join(result["test_case"]["solution_criteria"])

        score = result["score"]
        if score >= 8:
            score_class = "score-high"
        elif score <= 5:
            score_class = "score-low"
        else:
            score_class = "score-medium"

        html += f"""
            <tr>
                <td>{result["test_case"]["scenario"]}</td>
                <td class="prompt-inputs">{prompt_inputs_html}</td>
                <td class="criteria">• {criteria_string}</td>
                <td class="output"><pre>{result["output"]}</pre></td>
                <td class="score-col"><span class="score {score_class}">{score}</span></td>
                <td class="reasoning">{result["reasoning"]}</td>
            </tr>
        """

    html += """
            </tbody>
        </table>
    </body>
    </html>
    """

    return html

# PromptEvaluator Implementation
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string, variables):
        placeholders = re.findall(r"{([^{}]+)}", template_string)

        result = template_string
        for placeholder in placeholders:
            if placeholder in variables:
                result = result.replace(
                    "{" + placeholder + "}", str(variables[placeholder])
                )

        return result.replace("{{", "{").replace("}}", "}")

    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        """Generate a list of unique ideas for test cases based on the task description"""

        prompt = """
        Generate {num_cases} unique, diverse ideas for testing a prompt that accomplishes this task:
        
        <task_description>
        {task_description}
        </task_description>

        The prompt will receive the following inputs
        <prompt_inputs>
        {prompt_inputs_spec}
        </prompt_inputs>
        
        Each idea should represent a distinct scenario or example that tests different aspects of the task.
        
        Output Format:
        Provide your response as a structured JSON array where each item is a brief description of the idea.
        
        Example:
        ```json
        [
            "Testing with technical computer science terminology",
            "Testing with medical research findings",
            "Testing with complex mathematical concepts",
            ...
        ]
        ```
        
        Ensure each idea is:
        - Clearly distinct from the others
        - Relevant to the task description
        - Specific enough to guide generation of a full test case
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output

        Remember, only generate {num_cases} unique ideas
        """

        system_prompt = "You are a test scenario designer specialized in creating diverse, unique testing scenarios."

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": str # {val},'

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=1.0,
        )

        return json.loads(text)

    def generate_test_case(self, task_description, idea, prompt_inputs_spec={}):
        """Generate a single test case based on the task description and a specific idea"""

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": "EXAMPLE_VALUE", // {val}\n'

        allowed_keys = ", ".join([f'"{key}"' for key in prompt_inputs_spec.keys()])

        prompt = """
        Generate a single detailed test case for a prompt evaluation based on:
        
        <task_description>
        {task_description}
        </task_description>
        
        <specific_idea>
        {idea}
        </specific_idea>
        
        <allowed_input_keys>
        {allowed_keys}
        </allowed_input_keys>
        
        Output Format:
        ```json
        {{
            "prompt_inputs": {{
            {example_prompt_inputs}
            }},
            "solution_criteria": ["criterion 1", "criterion 2", ...] // Concise list of criteria for evaluating the solution, 1 to 4 items
        }}
        ```
        
        IMPORTANT REQUIREMENTS:
        - You MUST ONLY use these exact input keys in your prompt_inputs: {allowed_keys}        
        - Do NOT add any additional keys to prompt_inputs
        - All keys listed in allowed_input_keys must be included in your response
        - Make the test case realistic and practically useful
        - Include measurable, concise solution criteria
        - The solution criteria should ONLY address the direct requirements of the task description and the generated prompt_inputs
        - Avoid over-specifying criteria with requirements that go beyond the core task
        - Keep solution criteria simple, focused, and directly tied to the fundamental task
        - The test case should be tailored to the specific idea provided
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output
        - DO NOT include any fields beyond those specified in the output format

        Here's an example of a sample input with an ideal output:
        <sample_input>
        <sample_task_description>
        Extract topics out of a passage of text
        </sample_task_description>
        <sample_specific_idea>
        Testing with a text that contains multiple nested topics and subtopics (e.g., a passage about renewable energy that covers solar power economics, wind turbine technology, and policy implications simultaneously)
        </sample_specific_idea>

        <sample_allowed_input_keys>
        "content"
        </sample_allowed_input_keys>
        </sample_input>
        <ideal_output>
        ```json
        {
            "prompt_inputs": {
                "content": "The transition to renewable energy encompasses numerous interdependent dimensions. Solar photovoltaic technology has seen dramatic cost reductions, with panel efficiency improving 24% since 2010 while manufacturing costs declined by 89%, making it economically competitive with fossil fuels in many markets. Concurrently, wind energy has evolved through innovative turbine designs featuring carbon-fiber composite blades and advanced control systems that increase energy capture by 35% in low-wind conditions."
            },
            "solution_criteria": [
                "Includes all topics mentioned"   
            ]
        }
        ```
        </ideal_output>
        This is ideal output because the solution criteria is concise and doesn't ask for anything outside of the scope of the task description.
        """

        system_prompt = "You are a test case creator specializing in designing evaluation scenarios."

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "allowed_keys": allowed_keys,
                "task_description": task_description,
                "idea": idea,
                "example_prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=0.7,
        )

        test_case = json.loads(text)
        test_case["task_description"] = task_description
        test_case["scenario"] = idea

        return test_case

    def generate_dataset(
        self,
        task_description,
        prompt_inputs_spec={},
        num_cases=1,
        output_file="dataset.json",
    ):
        """Generate test dataset based on task description and save to file"""
        ideas = self.generate_unique_ideas(
            task_description, prompt_inputs_spec, num_cases
        )

        dataset = []
        completed = 0
        total = len(ideas)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_idea = {
                executor.submit(
                    self.generate_test_case,
                    task_description,
                    idea,
                    prompt_inputs_spec,
                ): idea
                for idea in ideas
            }

            for future in concurrent.futures.as_completed(future_to_idea):
                try:
                    result = future.result()
                    completed += 1
                    current_percentage = int((completed / total) * 100)
                    milestone_percentage = (current_percentage // 20) * 20

                    if milestone_percentage > last_reported_percentage:
                        print(f"Generated {completed}/{total} test cases")
                        last_reported_percentage = milestone_percentage

                    dataset.append(result)
                except Exception as e:
                    print(f"Error generating test case: {e}")

        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2)

        return dataset

    def grade_output(self, test_case, output, extra_criteria):
        """Grade the output of a test case using the model"""

        prompt_inputs = ""
        for key, value in test_case["prompt_inputs"].items():
            val = value.replace("\n", "\\n")
            prompt_inputs += f'"{key}":"{val}",\n'

        extra_criteria_section = ""
        if extra_criteria:
            extra_criteria_template = """
            Mandatory Requirements - ANY VIOLATION MEANS AUTOMATIC FAILURE (score of 3 or lower):
            <extra_important_criteria>
            {extra_criteria}
            </extra_important_criteria>
            """
            extra_criteria_section = self.render(
                dedent(extra_criteria_template),
                {"extra_criteria": extra_criteria},
            )

        eval_template = """
        Your task is to evaluate the following AI-generated solution with EXTREME RIGOR.

        Original task description:
        <task_description>
        {task_description}
        </task_description>

        Original task inputs:
        <task_inputs>
        {{ {prompt_inputs} }}
        </task_inputs>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Criteria you should use to evaluate the solution:
        <criteria>
        {solution_criteria}
        </criteria>

        {extra_criteria_section}

        Scoring Guidelines:
        * Score 1-3: Solution fails to meet one or more MANDATORY requirements
        * Score 4-6: Solution meets all mandatory requirements but has significant deficiencies in secondary criteria
        * Score 7-8: Solution meets all mandatory requirements and most secondary criteria, with minor issues
        * Score 9-10: Solution meets all mandatory and secondary criteria

        IMPORTANT SCORING INSTRUCTIONS:
        * Grade the output based ONLY on the listed criteria. Do not add your own extra requirements.
        * If a solution meets all of the mandatory and secondary criteria give it a 10
        * Don't complain that the solution "only" meets the mandatory and secondary criteria. Solutions shouldn't go above and beyond - they should meet the exact listed criteria.
        * ANY violation of a mandatory requirement MUST result in a score of 3 or lower
        * The full 1-10 scale should be utilized - don't hesitate to give low scores when warranted

        Output Format
        Provide your evaluation as a structured JSON object with the following fields, in this specific order:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "score": A number between 1-10

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
        """

        eval_prompt = self.render(
            dedent(eval_template),
            {
                "task_description": test_case["task_description"],
                "prompt_inputs": prompt_inputs,
                "output": output,
                "solution_criteria": "\n".join(test_case["solution_criteria"]),
                "extra_criteria_section": extra_criteria_section,
            },
        )

        messages = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, "```json")
        eval_text = chat(
            messages,
            stop_sequences=["```"],
            temperature=0.0,
        )
        return json.loads(eval_text)

    def run_test_case(self, test_case, run_prompt_function, extra_criteria=None):
        """Run a test case and grade the result"""
        output = run_prompt_function(test_case["prompt_inputs"])

        model_grade = self.grade_output(test_case, output, extra_criteria)
        model_score = model_grade["score"]
        reasoning = model_grade["reasoning"]

        return {
            "output": output,
            "test_case": test_case,
            "score": model_score,
            "reasoning": reasoning,
        }

    def run_evaluation(
        self,
        run_prompt_function,
        dataset_file,
        extra_criteria=None,
        json_output_file="output.json",
        html_output_file="output.html",
    ):
        """Run evaluation on all test cases in the dataset"""
        with open(dataset_file, "r") as f:
            dataset = json.load(f)

        results = []
        completed = 0
        total = len(dataset)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_test_case = {
                executor.submit(
                    self.run_test_case,
                    test_case,
                    run_prompt_function,
                    extra_criteria,
                ): test_case
                for test_case in dataset
            }

            for future in concurrent.futures.as_completed(future_to_test_case):
                result = future.result()
                completed += 1
                current_percentage = int((completed / total) * 100)
                milestone_percentage = (current_percentage // 20) * 20

                if milestone_percentage > last_reported_percentage:
                    print(f"Graded {completed}/{total} test cases")
                    last_reported_percentage = milestone_percentage
                results.append(result)

        average_score = mean([result["score"] for result in results])
        print(f"Average score: {average_score}")

        with open(json_output_file, "w") as f:
            json.dump(results, f, indent=2)

        html = generate_prompt_evaluation_report(results)
        with open(html_output_file, "w", encoding="utf-8") as f:
            f.write(html)

        return results

## Prompt engineering

In [68]:
# Create an instance of PromptEvaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
evaluator = PromptEvaluator(max_concurrent_tasks=3)

In [69]:
dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete"
    },
    output_file="dataset.json",
    num_cases=3
)

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


In [70]:
def run_prompt(prompt_inputs):
    prompt = f"""
    What should this person eat?

    - Height: {prompt_inputs["height"]}
    - Weight: {prompt_inputs["weight"]}
    - Goal: {prompt_inputs["goal"]}
    - Dietary restrictions: {prompt_inputs["restrictions"]}
    """

    messages = []
    messages = add_user_message(messages, prompt)
    return chat(messages)

In [71]:
result = evaluator.run_evaluation(
    run_prompt_function=run_prompt, dataset_file="dataset.json", extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact goods, portions, and timing
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 4


## Being clear and direct

In [72]:
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restricions.

    - Height: {prompt_inputs["height"]}
    - Weight: {prompt_inputs["weight"]}
    - Goal: {prompt_inputs["goal"]}
    - Dietary restrictions: {prompt_inputs["restrictions"]}
    """

    messages = []
    messages = add_user_message(messages, prompt)
    return chat(messages)

In [73]:
result = evaluator.run_evaluation(
    run_prompt_function=run_prompt, dataset_file="dataset.json", extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact goods, portions, and timing
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 8


## Being specific

In [74]:
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restricions.

    - Height: {prompt_inputs["height"]}
    - Weight: {prompt_inputs["weight"]}
    - Goal: {prompt_inputs["goal"]}
    - Dietary restrictions: {prompt_inputs["restrictions"]}

    Guidelines:
    1. Include accurate daily calorie amount
    2. Show protein, fat, and carb amounts
    3. Specify when to eat each meal
    4. Use only food that fit restrictions
    5. List all portion sizes in grams
    6. Keep budget-friendly if mentioned
    """

    messages = []
    messages = add_user_message(messages, prompt)
    return chat(messages)

In [75]:
result = evaluator.run_evaluation(
    run_prompt_function=run_prompt, dataset_file="dataset.json", extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact goods, portions, and timing
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 7.666666666666667


## Structure with XML Tags

In [76]:
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restricions.

    <athlete_information>
    - Height: {prompt_inputs["height"]}
    - Weight: {prompt_inputs["weight"]}
    - Goal: {prompt_inputs["goal"]}
    - Dietary restrictions: {prompt_inputs["restrictions"]}
    </athlete_information>

    Guidelines:
    1. Include accurate daily calorie amount
    2. Show protein, fat, and carb amounts
    3. Specify when to eat each meal
    4. Use only food that fit restrictions
    5. List all portion sizes in grams
    6. Keep budget-friendly if mentioned
    """

    messages = []
    messages = add_user_message(messages, prompt)
    return chat(messages)

In [77]:
result = evaluator.run_evaluation(
    run_prompt_function=run_prompt, dataset_file="dataset.json", extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact goods, portions, and timing
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 7.333333333333333


## One-shot and few-shot prompting

In [78]:
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restricions.

    <athlete_information>
    - Height: {prompt_inputs["height"]}
    - Weight: {prompt_inputs["weight"]}
    - Goal: {prompt_inputs["goal"]}
    - Dietary restrictions: {prompt_inputs["restrictions"]}
    </athlete_information>

    Guidelines:
    1. Include accurate daily calorie amount
    2. Show protein, fat, and carb amounts
    3. Specify when to eat each meal
    4. Use only food that fit restrictions
    5. List all portion sizes in grams
    6. Keep budget-friendly if mentioned

    Here is an example with a sample input and an ideal output:
    <sample_input>
    height: 178
    weight: 92
    goal: Powerlifter in cutting phase - lose 0.5kg per week while maintaining strength, target 2000 calories daily with high protein intake
    restrictions: No dairy products, must track macros precisely
    </sample_input>
    <ideal_output>
    # One-Day Meal Plan for Powerlifter (Cutting Phase)

    ## Daily Targets
    - **Total Calories:** 2000 kcal
    - **Protein:** 200g (40% - 800 kcal)
    - **Carbohydrates:** 175g (35% - 700 kcal)
    - **Fats:** 55g (25% - 500 kcal)

    ---

    ## Meal 1: Pre-Workout Breakfast (7:00 AM)
    - Oatmeal: 60g (dry weight)
    - Banana: 120g (1 medium)
    - Egg whites: 150g (5 large eggs)
    - Whole eggs: 50g (1 large egg)
    - Almond butter: 15g

    **Macros:** 520 kcal | 32g protein | 62g carbs | 15g fat

    ---

    ## Meal 2: Post-Workout Lunch (12:30 PM)
    - Chicken breast (grilled): 200g
    - White rice (cooked): 200g
    - Broccoli (steamed): 150g
    - Olive oil: 10g

    **Macros:** 625 kcal | 68g protein | 58g carbs | 12g fat

    ---

    ## Meal 3: Afternoon Snack (3:30 PM)
    - Tuna (canned in water, drained): 120g
    - Sweet potato (baked): 150g
    - Mixed greens salad: 100g
    - Walnuts: 15g

    **Macros:** 380 kcal | 35g protein | 32g carbs | 12g fat

    ---

    ## Meal 4: Dinner (7:00 PM)
    - Lean ground beef (93/7): 150g
    - Quinoa (cooked): 120g
    - Asparagus (grilled): 150g
    - Avocado: 40g

    **Macros:** 475 kcal | 45g protein | 23g carbs | 16g fat

    ---

    ## Daily Totals
    - **Calories:** 2000 kcal
    - **Protein:** 200g (40%)
    - **Carbohydrates:** 175g (35%)
    - **Fats:** 55g (25%)

    ---

    ## Additional Notes:
    - Drink 3-4 liters of water throughout the day
    - Take pre-workout 30 minutes before training (between Meals 1 & 2)
    - All cooking sprays are minimal and included in oil calculations
    - Weigh all foods raw unless specified as cooked
    - Consider a dairy-free protein powder if needed to hit protein targets on training days
    </ideal_output>
    This example solution fully satisfies all mandatory requirements: includes daily caloric total (2000 kcal), comprehensive macronutrient breakdown (200g protein, 175g carbs, 55g fat), and provides exact foods with specific portions and meal timing. It meets all secondary criteria: totals approximately 2000 calories, contains well over 180g protein (200g), excludes all dairy products, and includes detailed macro breakdowns for each meal. The plan is practical, well-structured for a powerlifter's schedule, and includes helpful additional notes about hydration and food preparation. The solution is compact yet comprehensive, addressing the athlete's specific cutting phase goals.


    """

    messages = []
    messages = add_user_message(messages, prompt)
    return chat(messages)

In [79]:
result = evaluator.run_evaluation(
    run_prompt_function=run_prompt, dataset_file="dataset.json", extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact goods, portions, and timing
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 7.666666666666667


## Exercise on prompting

In [80]:
dataset = evaluator.generate_dataset(
    task_description="Extract topics out of a passage of text from a scholarly article into a JSON array of strings",
    prompt_inputs_spec={
        "content": "One paragraph of text from a scholarly journal written in english",
    },
    output_file="dataset.json",
    num_cases=3
)

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


In [81]:
def run_prompt(prompt_inputs):
    prompt = f"""
    Extract each topic mentioned in the following article, and then return a JSON array of strings with each topic:

    <article>
    {prompt_inputs["content"]}
    </article>

    Guidelines:
    1. The output should only be a JSON array of strings and no more.
    2. The strings should contain only a topic without any extra commentary.
    3. You should avoid being redundant and having topic overlap

    Here is an example with the ideal output:
    <article>
    Quantum decoherence represents a critical challenge in maintaining coherent superposition states necessary for quantum computation. The interaction between a quantum system and its environment induces entanglement with environmental degrees of freedom, leading to the apparent collapse of the wavefunction through a process described by the Lindblad master equation. Recent advances in topological quantum error correction, particularly surface codes and Majorana fermion-based approaches, have demonstrated potential for fault-tolerant quantum gates with error thresholds approaching 1%. Meanwhile, quantum entanglement dynamics in many-body systems exhibit emergent phenomena such as thermalization and eigenstate thermalization hypothesis violations in integrable systems. The preservation of Bell state fidelity in ion trap architectures relies on sympathetic cooling techniques and dynamical decoupling sequences that mitigate dephasing from magnetic field fluctuations.
    </article>

    <output>
    ```json
    [
    "Quantum decoherence",
    "Quantum computation",
    "Wavefunction collapse",
    "Lindblad master equation",
    "Topological quantum error correction",
    "Surface codes",
    "Majorana fermions",
    "Fault-tolerant quantum gates",
    "Quantum entanglement dynamics",
    "Many-body systems",
    "Thermalization",
    "Eigenstate thermalization hypothesis",
    "Integrable systems",
    "Bell state fidelity",
    "Ion trap architectures",
    "Sympathetic cooling",
    "Dynamical decoupling",
    "Dephasing"
    ]
    ```

    The solution meets all mandatory requirements: it is a valid JSON array of strings, each string contains only a topic without commentary, and the response contains only the JSON array. It also exceeds all secondary criteria by extracting 18 topics (well above the minimum of 8), including both high-level topics and specialized subtopics as required. The topics are accurately extracted from the passage and appropriately formatted.
    """

    messages = []
    messages = add_user_message(messages, prompt)
    return chat(messages)

In [82]:
result = evaluator.run_evaluation(
    run_prompt_function=run_prompt, dataset_file="dataset.json", extra_criteria="""
    - Contains a JSON array of strings, containing each topic mentioned in the article.
    - The strings should contain only a topic without any extra commentary
    - Response should contain the JSON array and nothing else
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 9


## Tool use (How to set reminders)

We need three tools:
1. Get the current date time
2. Add duration to date time
3. Set remainders

In [83]:
# Tools and Schemas

from datetime import datetime, timedelta

def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [84]:
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format can not be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current date and time formatted according to the specified format string. Useful when you need to know the current date, time, or both in a specific format.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime format string that controls the output format. Defaults to '%Y-%m-%d %H:%M:%S' (e.g. '2026-06-14 15:30:00'). Common directives: %Y=4-digit year, %m=month, %d=day, %H=hour (24h), %M=minute, %S=second. Must not be empty.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
})

In [85]:
messages = []

messages.append(
    {
        "role": "user",
        "content": "What is the exact time, formatted as HH:MM:SS?"
    }
)

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

messages.append({
    "role": "assistant",
    "content": response.content
})

In [86]:
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)
    return messages

def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)
    return messages

def chat(messages, system=None, temperature=1.0, stop_sequences=None, tools=None):

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    if tools:
        params["tools"] = tools

    message = client.messages.create(
        **params
    )

    return message

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )


In [87]:
messages = []

messages = add_user_message(messages, "What's the current time in HH:MM:SS format?")

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

messages = add_assistant_message(messages, response.content)

In [88]:
response

Message(id='msg_01RbiQJ6aiQuGF8s2MywzgPA', container=None, content=[ToolUseBlock(id='toolu_01ReYWKxddYUSPyd5hFc1WYS', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=713, output_tokens=62, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [89]:
messages

[{'role': 'user', 'content': "What's the current time in HH:MM:SS format?"},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01ReYWKxddYUSPyd5hFc1WYS', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

## Implement multiturn conversation with tools

In [90]:
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)

def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    tools_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True
            }

        tools_result_blocks.append(tool_result_block)

    return tools_result_blocks
    
    

In [91]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema])

        messages = add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        messages = add_user_message(messages, tool_results)

    return messages


In [92]:
messages = []
messages = add_user_message(
    messages, "What is the current time in HH:MM format? Also, what is the current time in SS format?"
)
run_conversation(messages)


The current time is:
- **HH:MM format:** 14:23
- **SS format (seconds):** 45

So the full current time is 14:23:45.


[{'role': 'user',
  'content': 'What is the current time in HH:MM format? Also, what is the current time in SS format?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01SGxyiFkvhJkSne1kPFDnvw', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use'),
   ToolUseBlock(id='toolu_01JmyRh5T1M7Wbzoy4ujPVt3', caller=DirectCaller(type='direct'), input={'date_format': '%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01SGxyiFkvhJkSne1kPFDnvw',
    'content': '"14:23"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01JmyRh5T1M7Wbzoy4ujPVt3',
    'content': '"45"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time is:\n- **HH:MM format:** 14:23\n- **SS format (seconds):** 45\n\nSo the full current time is 14:23:45.', type='text')]}]

In [93]:
messages = []
messages = add_user_message(
    messages, "Set a reminder to go to medical exams one week from today"
)
run_conversation(messages)

I'll help you set a reminder for your medical exams one week from today. Let me first get the current date and time, then calculate the date one week from now.


----
Setting the following reminder for 2026-07-11T14:23:49:
Go to medical exams
----
Perfect! I've set a reminder for you to go to medical exams on **Saturday, July 11, 2026 at 2:23 PM** (one week from today). You'll receive a notification at that time.


[{'role': 'user',
  'content': 'Set a reminder to go to medical exams one week from today'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you set a reminder for your medical exams one week from today. Let me first get the current date and time, then calculate the date one week from now.", type='text'),
   ToolUseBlock(id='toolu_01PKZpUA6e35Af8cZ5wbfCWB', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01PKZpUA6e35Af8cZ5wbfCWB',
    'content': '"2026-07-04 14:23:49"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01ShMJ7NGmK3v2D4RkhFgPWj', caller=DirectCaller(type='direct'), input={'datetime_str': '2026-07-04 14:23:49', 'input_format': '%Y-%m-%d %H:%M:%S', 'duration': 1, 'unit': 'weeks'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_

## Fine-grained tool calling

In [98]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

# Helper functions


def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)


def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

# Tool definition
from anthropic.types import ToolParam

save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)
save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)


def save_article(**kwargs):
    return "Article saved!"

# Tool Running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

# Run conversation
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

In [101]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema],
)


>>> Tool Call: "save_article"
{"abstract": "A novel quantum-inspired deep learning architecture achieves state-of-the-art performance on graph neural network benchmarks through adaptive attention mechanisms.", "meta": {"word_count":3847,"review":"This paper introduces QuantumGraphNet, a hybrid architecture combining principles from quantum computing with modern deep learning techniques. The authors propose an adaptive attention mechanism that dynamically adjusts based on graph topology, showing significant improvements over traditional GNN approaches. Experimental results demonstrate a 15% improvement in node classification accuracy on standard benchmarks including Cora, CiteSeer, and PubMed datasets. The theoretical framework is well-grounded, drawing parallels between quantum superposition and multi-head attention. Ablation studies effectively isolate the contribution of each component. However, the computational overhead of the quantum-inspired operations may limit practical applic

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_016CVSw5eBU71YLLZx6yXfyy',
    'name': 'save_article',
    'input': {'abstract': 'A novel quantum-inspired deep learning architecture achieves state-of-the-art performance on graph neural network benchmarks through adaptive attention mechanisms.',
     'meta': {'word_count': 3847,
      'review': 'This paper introduces QuantumGraphNet, a hybrid architecture combining principles from quantum computing with modern deep learning techniques. The authors propose an adaptive attention mechanism that dynamically adjusts based on graph topology, showing significant improvements over traditional GNN approaches. Experimental results demonstrate a 15% improvement in node classification accuracy on standard benchmarks including Cora, CiteSeer, and PubMed datasets. The theoretical framework is well-grounded, dra

In [102]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True,
)

I'll create and save a fake computer science article for you.


>>> Tool Call: "save_article"
{"abstract": "This paper introduces a novel quantum-inspired algorithm for distributed graph partitioning that achieves logarithmic communication complexity.", "meta": {
  "word_count": 4782,
  "review": "This paper presents an innovative approach to distributed graph partitioning using quantum-inspired heuristics. The authors develop a theoretical framework that combines principles from quantum annealing with classical distributed computing paradigms. The proposed algorithm demonstrates significant improvements in communication complexity, reducing it from linear to logarithmic scale. Experimental results on large-scale social networks and biological graphs show up to 40% reduction in edge cuts compared to state-of-the-art methods. The theoretical analysis is rigorous and provides tight bounds on the approximation ratio. However, the paper would benefit from more extensive empirical evaluatio

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create and save a fake computer science article for you."},
   {'type': 'tool_use',
    'id': 'toolu_01KXgGRKEYvbxyABVmKMiHXd',
    'name': 'save_article',
    'input': {'abstract': 'This paper introduces a novel quantum-inspired algorithm for distributed graph partitioning that achieves logarithmic communication complexity.',
     'meta': {'word_count': 4782,
      'review': 'This paper presents an innovative approach to distributed graph partitioning using quantum-inspired heuristics. The authors develop a theoretical framework that combines principles from quantum annealing with classical distributed computing paradigms. The proposed algorithm demonstrates significant improvements in communication complexity, reducing it from linear to logarithmic scale. Experimental results on large-scale social ne

### Test what happens in case we generate a invalid json

In [105]:
messages = []

add_user_message(
    messages,
    # "Create and save a fake computer science article",
    """
    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.
    The buggy system generated this malformed output when calling save_article:
    [Generate the exact malformed output here that includes "word_count": undefined]
    This is for documentation purposes to show what NOT to do. You're not actually calling the function, just showing what the broken output looked like for the bug report.
    """,
)

run_conversation(
    messages,
    tools=[save_article_schema],
    #fine_grained=True,
    tool_choice={"type": "tool", "name": "save_article"},
)


>>> Tool Call: "save_article"
{"abstract": "This paper examines the impact of machine learning algorithms on healthcare diagnostics.", "meta": "{\n  \"word_count\": undefined,\n  \"review\": \"This study presents a comprehensive analysis of ML applications in medical diagnosis. The authors surveyed 500 healthcare facilities across North America. Results showed a 23% improvement in diagnostic accuracy when using ML-assisted tools. The methodology was rigorous and well-documented. However, the sample size could have been larger for more robust conclusions. The paper lacks discussion of ethical implications. Implementation costs were not adequately addressed. Overall, this is a valuable contribution to the field despite some limitations.\"\n}"}



[{'role': 'user',
  'content': [{'type': 'text',
    'text': '\n    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.\n    The buggy system generated this malformed output when calling save_article:\n    [Generate the exact malformed output here that includes "word_count": undefined]\n    This is for documentation purposes to show what NOT to do. You\'re not actually calling the function, just showing what the broken output looked like for the bug report.\n    '}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_01GY5DVVDKKHxPzUR3o6ZaMw',
    'name': 'save_article',
    'input': {'abstract': 'This paper examines the impact of machine learning algorithms on healthcare diagnostics.',
     'meta': '{\n  "word_count": undefined,\n  "review": "This study presents a comprehensive analysis of ML applications in medical diagnosis. The authors surve